In [10]:

import pandas as pd

df = pd.read_csv("../data/wasde/wasde_processed_all_historicals.csv")
# ============================================================
# 1. WASDE DISPLAY ORDER
# ============================================================

MONTH_ABBR = (
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
)


AREA_YIELD_ITEMS = [
    "area planted",
    "planted area",
    "planted",
    "area harvested",
    "harvested area",
    "harvested",
    "yield",
]

SUPPLY_ITEMS = [
    "beginning stocks",
    "production",
    "imports",
    "supply total",
    "total supply",
]

WASDE_ITEM_ORDER = {
    "corn": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "domestic feed or crush",
        "feed residual",
        "food seed industrial",
        "food feed industrial",
        "ethanol",
        "crushing",
        "crushings",
        "crush",
        "bio diesel",
        "seed",
        "food",
        "residual",
        "domestic total",
        "total domestic",
        "domestic use",
        "domestic disappearance",
        "exports",
        "exports total",
        "use total",
        "unaccounted",
        "loss",
        "total use",
        "ending stocks",
        "stock to use",
        "farm price",
        "avg. farm price",
        "avg. price",
        "average farm price",
        "season-average farm price",
    ],

    "soybeans": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "crushing",
        "crushings",
        "crush",
        "domestic feed or crush",
        "food seed industrial",
        "food feed industrial",
        "seed",
        "residual",
        "bio diesel",
        "domestic total",
        "total domestic",
        "domestic use",
        "domestic disappearance",
        "exports",
        "exports total",
        "use total",
        "unaccounted",
        "loss",
        "total use",
        "ending stocks",
        "stock to use",
        "avg. farm price",
        "avg. price",
    ],

    "wheat": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "food",
        "seed",
        "feed residual",
        "residual",
        "domestic total",
        "total domestic",
        "domestic use",
        "domestic disappearance",
        "exports",
        "exports total",
        "use total",
        "unaccounted",
        "loss",
        "total use",
        "ending stocks",
        "stock to use",
        "avg. farm price",
        "avg. price",
    ],

    "default": AREA_YIELD_ITEMS + SUPPLY_ITEMS + [
        "feed residual",
        "food seed industrial",
        "food feed industrial",
        "domestic total",
        "total domestic",
        "domestic use",
        "domestic disappearance",
        "exports",
        "exports total",
        "use total",
        "unaccounted",
        "loss",
        "total use",
        "ending stocks",
        "stock to use",
        "avg. farm price",
        "avg. price",
    ],
}



# ============================================================
# 2. WASDE DISPLAY LABELS
# ============================================================


WASDE_ITEM_LABELS = {
    # Area and yield
    "area planted": "Area Planted",
    "planted area": "Area Planted",
    "planted": "Area Planted",
    "area harvested": "Area Harvested",
    "harvested area": "Area Harvested",
    "harvested": "Area Harvested",
    "yield": "Yield",

    # Supply
    "beginning stocks": "Beginning Stocks",
    "production": "Production",
    "imports": "Imports",
    "supply total": "Total Supply",
    "total supply": "Total Supply",

    # Domestic use
    "domestic feed or crush": "Domestic Feed or Crush",
    "feed and residual": "Feed & Residual",
    "feed residual": "Feed & Residual",
    "food seed industrial": "Food, Seed & Industrial",
    "food seed and industrial": "Food, Seed & Industrial",
    "food feed industrial": "Food, Feed & Industrial",
    "ethanol": "Ethanol",
    "ethanol and by-products": "Ethanol & By-products",
    "crushing": "Crushing",
    "crushings": "Crushings",
    "crush": "Crush",
    "bio diesel": "Biodiesel",
    "food": "Food",
    "seed": "Seed",
    "residual": "Residual",
    "domestic total": "Domestic Use",
    "total domestic": "Domestic Use",
    "domestic use": "Domestic Use",
    "domestic disappearance": "Domestic Disappearance",

    # Exports and total use
    "exports": "Exports",
    "exports total": "Total Exports",
    "use total": "Total Use",
    "total use": "Total Use",
    "unaccounted": "Unaccounted",
    "loss": "Loss",

    # Ending stocks and prices
    "ending stocks": "Ending Stocks",
    "stock to use": "Stocks-to-Use",
    "farm price": "Avg. Farm Price",
    "avg. farm price": "Avg. Farm Price",
    "avg. price": "Avg. Farm Price",
    "average farm price": "Avg. Farm Price",
    "season-average farm price": "Avg. Farm Price",
}



# ============================================================
# 8. GROUP WASDE ROWS INTO FUNDAMENTAL SECTIONS
# ============================================================


WASDE_SECTIONS = {
    "Area & Yield": [
        "Area Planted",
        "Area Harvested",
        "Yield",
    ],

    "Supply": [
        "Beginning Stocks",
        "Production",
        "Imports",
        "Total Supply",
    ],

    "Domestic Use": [
        "Domestic Feed or Crush",
        "Feed & Residual",
        "Food, Seed & Industrial",
        "Food, Feed & Industrial",
        "Food",
        "Seed",
        "Residual",
        "Ethanol",
        "Ethanol & By-products",
        "Crushing",
        "Crushings",
        "Crush",
        "Biodiesel",
        "Domestic Use",
        "Domestic Disappearance",
        "Unaccounted",
        "Loss",
    ],

    "Exports": [
        "Exports",
        "Total Exports",
    ],

    "Total Use & Ending Stocks": [
        "Total Use",
        "Ending Stocks",
        "Stocks-to-Use",
        "Avg. Farm Price",
    ],
}



def split_wasde_table(table):
    """Split a formatted WASDE table into ordered sections."""
    sections = {}
    assigned_positions = set()

    for section_name, item_labels in WASDE_SECTIONS.items():
        positions = [
            i
            for i, label in enumerate(table.index)
            if label in item_labels
        ]

        if positions:
            sections[section_name] = table.iloc[positions].copy()
            assigned_positions.update(positions)

    # Keep any unrecognized items instead of dropping them.
    other_positions = [
        i
        for i in range(len(table))
        if i not in assigned_positions
    ]

    if other_positions:
        sections["Other Items"] = table.iloc[other_positions].copy()

    return sections

def format_wasde_item(item_name):
    """Convert raw WASDE item names to display labels."""
    item_name = str(item_name).strip().lower()

    return WASDE_ITEM_LABELS.get(
        item_name,
        item_name.replace("_", " ").title(),
    )


# ============================================================
# 3. ORDERING AND LABEL FORMATTERS
# ============================================================

def order_wasde_items(item_names, commodity_name=None):
    """Apply configured order and preserve unknown-item order."""
    commodity_name = (
        str(commodity_name).strip().lower()
        if commodity_name is not None
        else None
    )

    item_order = WASDE_ITEM_ORDER.get(
        commodity_name,
        WASDE_ITEM_ORDER["default"],
    )

    item_ranks = {
        item_name: position
        for position, item_name in enumerate(item_order)
    }

    # Stable sorting keeps unknown items in their original order.
    return sorted(
        item_names,
        key=lambda item_name: item_ranks.get(
            str(item_name).strip().lower(),
            len(item_ranks),
        ),
    )


def parse_month_offset(month_offset):
    """Convert '1_10' into (1, 10)."""
    year_offset, calendar_month = str(month_offset).split("_")
    return int(year_offset), int(calendar_month)


def format_month_offset(month_offset, crop_year=None):
    """Format a month offset as 'Oct-26' or 'Oct (Y1)'."""
    year_offset, calendar_month = parse_month_offset(month_offset)
    month_name = MONTH_ABBR[calendar_month - 1]

    if crop_year is None:
        return f"{month_name} (Y{year_offset})"

    calendar_year = int(crop_year) + year_offset
    return f"{month_name}-{calendar_year % 100:02d}"


def format_marketing_year(crop_year, show_full_year=False):
    """Convert 2025 to '2025/26' or '2025/2026'."""
    crop_year = int(crop_year)
    next_year = crop_year + 1

    if show_full_year:
        return f"{crop_year}/{next_year}"

    return f"{crop_year}/{next_year % 100:02d}"


# ============================================================
# 4. PREPARE AND FILTER DATA
# ============================================================

def prepare_wasde_data(dataframe):
    """Standardize WASDE column names and data types."""
    prepared_data = dataframe.copy()

    prepared_data.columns = [
        str(column_name).strip()
        for column_name in prepared_data.columns
    ]

    prepared_data = prepared_data.rename(columns={
        "Report Date": "report_date",
        "Value": "value",
        "crop year": "crop_year",
        "month offset": "month_offset",
    })

    required_columns = [
        "report_date", "value", "crop_year", "month_offset",
        "commodity", "country", "snd_item",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in prepared_data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required WASDE columns: {missing_columns}"
        )

    prepared_data["report_date"] = pd.to_datetime(
        prepared_data["report_date"],
        errors="coerce",
    )

    prepared_data["value"] = pd.to_numeric(
        prepared_data["value"],
        errors="coerce",
    )

    prepared_data["crop_year"] = pd.to_numeric(
        prepared_data["crop_year"],
        errors="coerce",
    )

    text_columns = [
        "commodity",
        "country",
        "snd_item",
        "wheat_class",
        "crop",
        "table",
    ]

    for column_name in text_columns:
        if column_name in prepared_data.columns:
            prepared_data[column_name] = (
                prepared_data[column_name]
                .astype("string")
                .str.strip()
                .str.lower()
            )

    month_offset_parts = (
        prepared_data["month_offset"]
        .astype("string")
        .str.split("_", n=1, expand=True)
    )

    prepared_data["year_off"] = pd.to_numeric(
        month_offset_parts[0],
        errors="coerce",
    )

    prepared_data["month_num"] = pd.to_numeric(
        month_offset_parts[1],
        errors="coerce",
    )

    prepared_data = prepared_data.dropna(
        subset=[
            "report_date",
            "value",
            "crop_year",
            "year_off",
            "month_num",
        ]
    ).copy()

    for column_name in ["crop_year", "year_off", "month_num"]:
        prepared_data[column_name] = (
            prepared_data[column_name].astype(int)
        )

    prepared_data["month_offset"] = (
        prepared_data["year_off"].astype(str)
        + "_"
        + prepared_data["month_num"].astype(str).str.zfill(2)
    )

    return prepared_data


def select_wasde_data(
    dataframe,
    commodity,
    country,
    table=None,
    wheat_class=None,
):
    """Filter by commodity, country and optional table/class."""
    selected_data = dataframe[
        dataframe["commodity"].eq(commodity.lower()).fillna(False)
        & dataframe["country"].eq(country.lower()).fillna(False)
    ].copy()

    if table and "table" in selected_data.columns:
        selected_data = selected_data[
            selected_data["table"].eq(table.lower()).fillna(False)
        ]

    if wheat_class and "wheat_class" in selected_data.columns:
        selected_data = selected_data[
            selected_data["wheat_class"]
            .eq(wheat_class.lower())
            .fillna(False)
        ]

    return selected_data


# ============================================================
# 5. SHARED TABLE HELPERS
# ============================================================

def get_latest_records(dataframe, key_columns):
    """Keep the latest report for each combination of keys."""
    return (
        dataframe.sort_values("report_date")
        .drop_duplicates(subset=key_columns, keep="last")
    )


def get_commodity_name(dataframe):
    """Return the selected commodity name."""
    commodity_names = dataframe["commodity"].dropna()

    if commodity_names.empty:
        return None

    return str(commodity_names.iloc[0]).strip().lower()


def order_balance_sheet_rows(balance_sheet, commodity_name=None):
    """Order raw item names before applying display labels."""
    ordered_item_names = order_wasde_items(
        balance_sheet.index.tolist(),
        commodity_name,
    )

    return balance_sheet.reindex(ordered_item_names)


def style_wasde_table(table):
    """Display numeric values to 2 decimal places."""
    return table.style.format("{:.2f}", na_rep="—")


# ============================================================
# 6. TABLE 1: YEARLY BALANCE SHEET
# ============================================================

def build_yearly_table(
    dataframe,
    as_of="latest",
    first_year=None,
    last_year=None,
):
    """Rows are WASDE items; columns are marketing years."""
    yearly_data = dataframe.copy()

    if as_of != "latest":
        year_offset, calendar_month = parse_month_offset(as_of)

        yearly_data = yearly_data[
            yearly_data["year_off"].eq(year_offset)
            & yearly_data["month_num"].eq(calendar_month)
        ]

    if first_year is not None:
        yearly_data = yearly_data[
            yearly_data["crop_year"] >= first_year
        ]

    if last_year is not None:
        yearly_data = yearly_data[
            yearly_data["crop_year"] <= last_year
        ]

    if yearly_data.empty:
        return pd.DataFrame()

    commodity_name = get_commodity_name(yearly_data)

    latest_records = get_latest_records(
        yearly_data,
        ["snd_item", "crop_year"],
    )

    balance_sheet = latest_records.pivot(
        index="snd_item",
        columns="crop_year",
        values="value",
    )

    # Sort using raw WASDE names first.
    balance_sheet = order_balance_sheet_rows(
        balance_sheet,
        commodity_name,
    )

    # Sort years chronologically before formatting their labels.
    balance_sheet = balance_sheet.reindex(
        sorted(balance_sheet.columns),
        axis=1,
    )

    balance_sheet.columns = [
        format_marketing_year(crop_year)
        for crop_year in balance_sheet.columns
    ]

    # Apply display labels only after sorting.
    balance_sheet.index = [
        format_wasde_item(item_name)
        for item_name in balance_sheet.index
    ]

    balance_sheet.index.name = "WASDE Item"
    balance_sheet.columns.name = "Marketing Year"

    return balance_sheet


# ============================================================
# 7. TABLE 2: MONTHLY BALANCE SHEET
# ============================================================

def build_monthly_table(
    dataframe,
    crop_year,
    crop_status=None,
):
    """Rows are WASDE items; columns are report months."""
    monthly_data = dataframe[
        dataframe["crop_year"].eq(int(crop_year))
    ].copy()

    if crop_status and "crop" in monthly_data.columns:
        monthly_data = monthly_data[
            monthly_data["crop"].eq(crop_status.lower())
        ]

    if monthly_data.empty:
        return pd.DataFrame()

    commodity_name = get_commodity_name(monthly_data)

    latest_records = get_latest_records(
        monthly_data,
        ["snd_item", "month_offset"],
    )

    month_order = (
        latest_records[
            ["month_offset", "year_off", "month_num"]
        ]
        .drop_duplicates()
        .sort_values(["year_off", "month_num"])
        ["month_offset"]
        .tolist()
    )

    balance_sheet = latest_records.pivot(
        index="snd_item",
        columns="month_offset",
        values="value",
    )

    balance_sheet = balance_sheet.reindex(columns=month_order)

    # Sort using raw WASDE names first.
    balance_sheet = order_balance_sheet_rows(
        balance_sheet,
        commodity_name,
    )

    # Format month headers after chronological ordering.
    balance_sheet.columns = [
        format_month_offset(month_offset, crop_year)
        for month_offset in month_order
    ]

    # Apply display labels after row ordering.
    balance_sheet.index = [
        format_wasde_item(item_name)
        for item_name in balance_sheet.index
    ]

    balance_sheet.index.name = "WASDE Item"
    balance_sheet.columns.name = (
        f"Marketing Year {format_marketing_year(crop_year)}"
    )

    return balance_sheet


wasde = prepare_wasde_data(df)

corn_us = select_wasde_data(
    wasde,
    commodity="corn",
    country="united states",
)

# Inspect raw item names
print(sorted(corn_us["snd_item"].dropna().unique()))


# ============================================================
# 1. YEARLY BALANCE SHEET — GROUPED BY SECTION
# ============================================================

yearly_table = build_yearly_table(corn_us)

yearly_sections = split_wasde_table(yearly_table)

print("YEARLY BALANCE SHEET")

for section_name, section_table in yearly_sections.items():
    print(f"\n{section_name}")
    display(style_wasde_table(section_table))


# ============================================================
# 2. MONTHLY BALANCE SHEET — GROUPED BY SECTION
# ============================================================

monthly_table = build_monthly_table(
    corn_us,
    crop_year=2026,
)

monthly_sections = split_wasde_table(monthly_table)

print("\nMONTHLY BALANCE SHEET — 2026 CROP YEAR")

for section_name, section_table in monthly_sections.items():
    print(f"\n{section_name}")
    display(style_wasde_table(section_table))


['avg. farm price', 'beginning stocks', 'domestic feed or crush', 'domestic total', 'ending stocks', 'ethanol', 'exports', 'feed residual', 'food seed industrial', 'harvested area', 'imports', 'planted area', 'production', 'stock to use', 'supply total', 'use total', 'yield']
YEARLY BALANCE SHEET

Area & Yield


Marketing Year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,,
Area Planted,86.40,88.20,91.90,97.20,95.40,90.60,88.00,94.00,90.20,88.90,89.70,90.70,93.30,88.20,94.60,90.90,98.80,96.80
Area Harvested,79.50,81.40,84.00,87.40,87.50,83.10,80.80,86.70,82.70,81.30,81.30,82.30,85.30,78.70,86.50,83.00,91.30,88.50
Yield,164.70,152.80,147.20,123.40,158.10,171.00,168.40,174.60,176.60,176.40,167.50,171.40,176.70,173.40,177.30,179.30,186.50,178.50



Supply


Marketing Year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,,
Beginning Stocks,1673.00,1708.00,28.64,989.00,821.00,1232.00,1731.00,1737.00,2293.00,2140.00,2221.00,1919.00,1235.00,1377.00,1360.00,1763.00,1551.00,1922.00
Production,13092.00,316.17,12360.00,10780.00,13829.00,14216.00,13602.00,15148.00,14609.00,14340.00,13620.00,14111.00,15074.00,13651.00,15341.00,14892.00,17021.00,15800.00
Imports,8.00,28.00,29.00,162.00,36.00,32.00,67.00,57.00,36.00,28.00,42.00,24.00,24.00,39.00,28.00,22.00,30.00,25.00
Total Supply,14774.00,14182.00,13516.00,11932.00,14686.00,15479.00,15401.00,16942.00,16939.00,16509.00,15883.00,16055.00,16333.00,15066.00,16729.00,16677.00,18602.00,17747.00



Domestic Use


Marketing Year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,,
Domestic Feed or Crush,130.57,121.74,115.46,110.06,127.92,135.23,130.05,139.01,134.73,137.93,149.80,142.20,145.31,139.36,147.46,138.55,161.30,151.14
Feed & Residual,5140.00,4793.00,4545.00,4333.00,5036.00,5324.00,5120.00,5472.00,5304.00,5430.00,5897.00,5598.00,5721.00,5486.00,5805.00,5454.00,6350.00,5950.00
"Food, Seed & Industrial",5939.00,6428.00,6439.00,6046.00,6501.00,6560.00,6646.00,6883.00,7056.00,6793.00,6287.00,6470.00,6764.00,6558.00,6868.00,6813.00,6905.00,6955.00
Ethanol,4568.00,5021.00,5011.00,4648.00,5134.00,5200.00,5224.00,5432.00,5605.00,5378.00,4857.00,5033.00,5326.00,5176.00,5478.00,5436.00,5550.00,5600.00
Domestic Use,11079.00,11220.00,279.02,10379.00,11537.00,11883.00,11766.00,12356.00,12360.00,12223.00,12185.00,12068.00,12484.00,12045.00,12673.00,12267.00,13255.00,12905.00


... (10 outputs left out)


Exports


Marketing Year 2026/27,May-26,Jun-26,Jul-26,Aug-26,Sep-26
WASDE Item,,,,,
Exports,3150.00,3150.00,3200.00,3275.00,3275.00



Total Use & Ending Stocks


Marketing Year 2026/27,May-26,Jun-26,Jul-26,Aug-26,Sep-26
WASDE Item,,,,,
Total Use,16205.00,16205.00,16255.00,16330.00,16180.00
Ending Stocks,1957.00,1960.00,1790.00,41.98,39.79
Stocks-to-Use,0.12,0.12,0.11,0.10,0.10
Avg. Farm Price,4.40,4.40,4.40,4.50,4.80


In [11]:
df["snd_item"].unique()

<ArrowStringArray>
[      'beginning stocks',           'domestic use',          'ending stocks',
                'exports',          'feed residual',                   'food',
                'imports',             'production',                   'seed',
           'supply total',              'use total',           'stock to use',
         'harvested area',           'planted area',         'domestic total',
                'ethanol',   'food seed industrial',                  'yield',
             'bio diesel',   'food feed industrial',               'crushing',
               'residual', 'domestic feed or crush',        'avg. farm price',
         'total domestic', 'domestic disappearance',             'avg. price',
                'planted',              'harvested',          'exports total',
            'unaccounted',                   'loss']
Length: 32, dtype: str

In [14]:

import pandas as pd

# ============================================================
# 1. LOAD DATA
# ============================================================

FILE_PATH = "../data/wasde/wasde_processed_all_historicals.csv"
df = pd.read_csv(FILE_PATH)

MONTH_ABBR = (
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
)


# ============================================================
# 2. WASDE DISPLAY LABELS
# Keys = raw identifiers; values = display labels
# ============================================================

WASDE_ITEM_LABELS = {
    # Area & Yield
    "planted area": "Area Planted",
    "planted": "Planted",
    "harvested area": "Area Harvested",
    "harvested": "Harvested",
    "yield": "Yield",

    # Supply
    "beginning stocks": "Beginning Stocks",
    "production": "Production",
    "imports": "Imports",
    "supply total": "Total Supply",

    # Domestic Use
    "domestic use": "Domestic Use",
    "domestic total": "Domestic Total",
    "total domestic": "Total Domestic",
    "domestic disappearance": "Domestic Disappearance",
    "domestic feed or crush": "Domestic Feed or Crush",
    "feed residual": "Feed & Residual",
    "food": "Food",
    "seed": "Seed",
    "food seed industrial": "Food, Seed & Industrial",
    "food feed industrial": "Food, Feed & Industrial",
    "ethanol": "Ethanol",
    "crushing": "Crushing",
    "bio diesel": "Biodiesel",
    "residual": "Residual",

    # Exports
    "exports": "Exports",
    "exports total": "Total Exports",

    # Total Use, Ending Stocks & Prices
    "use total": "Total Use",
    "ending stocks": "Ending Stocks",
    "stock to use": "Stocks-to-Use",
    "avg. farm price": "Avg. Farm Price",
    "avg. price": "Avg. Price",

    # Other
    "unaccounted": "Unaccounted",
    "loss": "Loss",
}


# ============================================================
# 3. WASDE SECTIONS
# Keys are raw identifiers, not display labels
# ============================================================

WASDE_SECTIONS = {
    "Area & Yield": [
        "planted area",
        "planted",
        "harvested area",
        "harvested",
        "yield",
    ],

    "Supply": [
        "beginning stocks",
        "production",
        "imports",
        "supply total",
    ],

    "Domestic Use": [
        "domestic feed or crush",
        "feed residual",
        "food seed industrial",
        "food feed industrial",
        "food",
        "seed",
        "ethanol",
        "crushing",
        "bio diesel",
        "residual",
        "domestic use",
        "domestic total",
        "total domestic",
        "domestic disappearance",
        "unaccounted",
        "loss",
    ],

    "Exports": [
        "exports",
        "exports total",
    ],

    "Total Use & Ending Stocks": [
        "use total",
        "ending stocks",
        "stock to use",
        "avg. farm price",
        "avg. price",
    ],
}


# ============================================================
# 4. PREPARE DATA
# ============================================================

def prepare_wasde_data(dataframe):
    """Standardize columns and data types."""

    data = dataframe.copy()

    # Normalize column names.
    data.columns = (
        data.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    # Handle known source column names after normalization.
    rename_columns = {
        "report_date": "report_date",
        "value": "value",
        "crop_year": "crop_year",
        "month_offset": "month_offset",
    }
    data = data.rename(columns=rename_columns)

    required_columns = [
        "report_date",
        "value",
        "crop_year",
        "month_offset",
        "commodity",
        "country",
        "snd_item",
    ]

    missing = [
        col for col in required_columns
        if col not in data.columns
    ]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    data["report_date"] = pd.to_datetime(
        data["report_date"], errors="coerce"
    )
    data["value"] = pd.to_numeric(
        data["value"], errors="coerce"
    )
    data["crop_year"] = pd.to_numeric(
        data["crop_year"], errors="coerce"
    )

    # Normalize text fields.
    text_columns = [
        "commodity", "country", "snd_item",
        "wheat_class", "crop", "table",
    ]

    for col in text_columns:
        if col in data.columns:
            data[col] = (
                data[col].astype("string").str.strip().str.lower()
            )

    # Parse month offset, e.g. 1_10 -> year offset 1, month 10.
    offset_parts = (
        data["month_offset"]
        .astype("string")
        .str.strip()
        .str.split("_", n=1, expand=True)
    )

    if offset_parts.shape[1] < 2:
        raise ValueError(
            "month_offset must use the format 'year_month', e.g. '1_10'."
        )

    data["year_off"] = pd.to_numeric(
        offset_parts[0], errors="coerce"
    )
    data["month_num"] = pd.to_numeric(
        offset_parts[1], errors="coerce"
    )

    data = data.dropna(
        subset=[
            "report_date", "value", "crop_year",
            "year_off", "month_num",
        ]
    ).copy()

    for col in ["crop_year", "year_off", "month_num"]:
        data[col] = data[col].astype(int)

    data["month_offset"] = (
        data["year_off"].astype(str)
        + "_"
        + data["month_num"].astype(str).str.zfill(2)
    )

    return data


# ============================================================
# 5. FILTER DATA
# ============================================================

def select_wasde_data(
    dataframe,
    commodity,
    country,
    table=None,
    wheat_class=None,
    crop_status=None,
):
    """Filter commodity, country and optional WASDE dimensions."""

    data = dataframe[
        dataframe["commodity"].eq(commodity.strip().lower()).fillna(False)
        & dataframe["country"].eq(country.strip().lower()).fillna(False)
    ].copy()

    if table is not None and "table" in data.columns:
        data = data[
            data["table"].eq(table.strip().lower()).fillna(False)
        ]

    if wheat_class is not None and "wheat_class" in data.columns:
        data = data[
            data["wheat_class"]
            .eq(wheat_class.strip().lower())
            .fillna(False)
        ]

    if crop_status is not None and "crop" in data.columns:
        data = data[
            data["crop"].eq(crop_status.strip().lower()).fillna(False)
        ]

    return data


# ============================================================
# 6. SHARED HELPERS
# ============================================================

def format_wasde_item(item_name):
    """Return display label for a raw WASDE identifier."""

    raw_item = str(item_name).strip().lower()

    return WASDE_ITEM_LABELS.get(
        raw_item,
        raw_item.replace("_", " ").title(),
    )


def format_marketing_year(crop_year, show_full_year=False):
    """Convert 2025 to 2025/26 or 2025/2026."""

    year = int(crop_year)
    next_year = year + 1

    if show_full_year:
        return f"{year}/{next_year}"

    return f"{year}/{next_year % 100:02d}"


def parse_month_offset(month_offset):
    """Convert '1_10' to (1, 10)."""

    year_offset, month = str(month_offset).split("_")
    return int(year_offset), int(month)


def format_month_offset(month_offset, crop_year=None):
    """Format offset as 'Oct (Y1)' or 'Oct-26'."""

    year_offset, month = parse_month_offset(month_offset)
    month_name = MONTH_ABBR[month - 1]

    if crop_year is None:
        return f"{month_name} (Y{year_offset})"

    calendar_year = int(crop_year) + year_offset
    return f"{month_name}-{calendar_year % 100:02d}"


def get_latest_records(dataframe, key_columns):
    """Keep the latest report for each combination of keys."""

    return (
        dataframe.sort_values("report_date")
        .drop_duplicates(subset=key_columns, keep="last")
    )


def get_commodity_name(dataframe):
    """Get commodity name for row ordering, if needed."""

    values = dataframe["commodity"].dropna()

    return str(values.iloc[0]).lower() if not values.empty else None


def style_wasde_table(table):
    """Format numeric values to two decimal places."""

    return table.style.format("{:.2f}", na_rep="—")


# ============================================================
# 7. YEARLY BALANCE SHEET
# ============================================================

def build_yearly_table(
    dataframe,
    as_of="latest",
    first_year=None,
    last_year=None,
):
    """
    Rows = raw WASDE identifiers.
    Columns = marketing years.

    as_of='latest' uses the latest available report for each item/year.
    To compare a consistent monthly offset, pass e.g. as_of='1_10'.
    """

    data = dataframe.copy()

    if as_of != "latest":
        year_offset, month = parse_month_offset(as_of)

        data = data[
            data["year_off"].eq(year_offset)
            & data["month_num"].eq(month)
        ]

    if first_year is not None:
        data = data[data["crop_year"] >= int(first_year)]

    if last_year is not None:
        data = data[data["crop_year"] <= int(last_year)]

    if data.empty:
        return pd.DataFrame()

    latest = get_latest_records(
        data,
        ["snd_item", "crop_year"],
    )

    table = latest.pivot(
        index="snd_item",
        columns="crop_year",
        values="value",
    )

    table = table.reindex(
        columns=sorted(table.columns)
    )

    table.columns = [
        format_marketing_year(year)
        for year in table.columns
    ]

    table.index.name = "WASDE Item"
    table.columns.name = "Marketing Year"

    return table


# ============================================================
# 8. MONTHLY BALANCE SHEET
# ============================================================

def build_monthly_table(
    dataframe,
    crop_year,
    crop_status=None,
):
    """
    Rows = raw WASDE identifiers.
    Columns = WASDE report months for one crop year.
    """

    data = dataframe[
        dataframe["crop_year"].eq(int(crop_year))
    ].copy()

    if crop_status is not None and "crop" in data.columns:
        data = data[
            data["crop"].eq(crop_status.strip().lower()).fillna(False)
        ]

    if data.empty:
        return pd.DataFrame()

    latest = get_latest_records(
        data,
        ["snd_item", "month_offset"],
    )

    month_order = (
        latest[["month_offset", "year_off", "month_num"]]
        .drop_duplicates()
        .sort_values(["year_off", "month_num"])
        ["month_offset"]
        .tolist()
    )

    table = latest.pivot(
        index="snd_item",
        columns="month_offset",
        values="value",
    )

    table = table.reindex(columns=month_order)

    table.columns = [
        format_month_offset(offset, crop_year)
        for offset in month_order
    ]

    table.index.name = "WASDE Item"
    table.columns.name = (
        f"Marketing Year {format_marketing_year(crop_year)}"
    )

    return table


# ============================================================
# 9. SPLIT TABLE INTO SECTIONS
# IMPORTANT: Group by raw identifiers before renaming.
# ============================================================

def split_wasde_table(table):
    """Split a raw-item-indexed table into configured sections."""

    sections = {}
    assigned_items = set()

    for section_name, raw_items in WASDE_SECTIONS.items():
        present_items = [
            item for item in raw_items
            if item in table.index
        ]

        if present_items:
            section_table = table.loc[present_items].copy()

            # Rename rows only for display.
            section_table.index = [
                format_wasde_item(item)
                for item in section_table.index
            ]
            section_table.index.name = "WASDE Item"

            sections[section_name] = section_table
            assigned_items.update(present_items)

    other_items = [
        item for item in table.index
        if item not in assigned_items
    ]

    if other_items:
        other_table = table.loc[other_items].copy()
        other_table.index = [
            format_wasde_item(item)
            for item in other_table.index
        ]
        other_table.index.name = "WASDE Item"

        sections["Other Items"] = other_table

    return sections


# ============================================================
# 10. RUN
# ============================================================

wasde = prepare_wasde_data(df)

corn_us = select_wasde_data(
    wasde,
    commodity="corn",
    country="united states",
)


# Inspect available raw identifiers.
# print("Available raw WASDE items:")
# print(sorted(corn_us["snd_item"].dropna().unique()))

# # Verify the 32-item mapping against the selected dataset.
# raw_items = set(corn_us["snd_item"].dropna().unique())
# mapped_items = set(WASDE_ITEM_LABELS)
# section_items = {
#     item
#     for items in WASDE_SECTIONS.values()
#     for item in items
# }

# print("\nMapping checks")
# print("Raw items in dataset:", len(raw_items))
# print("Keys in display mapping:", len(mapped_items))
# print("Items in sections:", len(section_items))
# print("Missing display labels:", sorted(raw_items - mapped_items))
# print("Missing from sections:", sorted(raw_items - section_items))
# print("Section identifiers not in dataset:", sorted(section_items - raw_items))

# ------------------------------------------------------------
# YEARLY BALANCE SHEET
# ------------------------------------------------------------

yearly_table = build_yearly_table(corn_us)

print("\nYEARLY BALANCE SHEET")

for section_name, section_table in split_wasde_table(yearly_table).items():
    print(f"\n{section_name}")
    display(style_wasde_table(section_table))

# ------------------------------------------------------------
# MONTHLY BALANCE SHEET
# ------------------------------------------------------------

monthly_table = build_monthly_table(
    corn_us,
    crop_year=2026,
)

print("\nMONTHLY BALANCE SHEET — 2026 CROP YEAR")

for section_name, section_table in split_wasde_table(monthly_table).items():
    print(f"\n{section_name}")
    display(style_wasde_table(section_table))



YEARLY BALANCE SHEET

Area & Yield


Marketing Year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,,
Area Planted,86.40,88.20,91.90,97.20,95.40,90.60,88.00,94.00,90.20,88.90,89.70,90.70,93.30,88.20,94.60,90.90,98.80,96.80
Area Harvested,79.50,81.40,84.00,87.40,87.50,83.10,80.80,86.70,82.70,81.30,81.30,82.30,85.30,78.70,86.50,83.00,91.30,88.50
Yield,164.70,152.80,147.20,123.40,158.10,171.00,168.40,174.60,176.60,176.40,167.50,171.40,176.70,173.40,177.30,179.30,186.50,178.50



Supply


Marketing Year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,,
Beginning Stocks,1673.00,1708.00,28.64,989.00,821.00,1232.00,1731.00,1737.00,2293.00,2140.00,2221.00,1919.00,1235.00,1377.00,1360.00,1763.00,1551.00,1922.00
Production,13092.00,316.17,12360.00,10780.00,13829.00,14216.00,13602.00,15148.00,14609.00,14340.00,13620.00,14111.00,15074.00,13651.00,15341.00,14892.00,17021.00,15800.00
Imports,8.00,28.00,29.00,162.00,36.00,32.00,67.00,57.00,36.00,28.00,42.00,24.00,24.00,39.00,28.00,22.00,30.00,25.00
Total Supply,14774.00,14182.00,13516.00,11932.00,14686.00,15479.00,15401.00,16942.00,16939.00,16509.00,15883.00,16055.00,16333.00,15066.00,16729.00,16677.00,18602.00,17747.00



Domestic Use


Marketing Year,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26,2026/27
WASDE Item,,,,,,,,,,,,,,,,,,
Domestic Feed or Crush,130.57,121.74,115.46,110.06,127.92,135.23,130.05,139.01,134.73,137.93,149.80,142.20,145.31,139.36,147.46,138.55,161.30,151.14
Feed & Residual,5140.00,4793.00,4545.00,4333.00,5036.00,5324.00,5120.00,5472.00,5304.00,5430.00,5897.00,5598.00,5721.00,5486.00,5805.00,5454.00,6350.00,5950.00
"Food, Seed & Industrial",5939.00,6428.00,6439.00,6046.00,6501.00,6560.00,6646.00,6883.00,7056.00,6793.00,6287.00,6470.00,6764.00,6558.00,6868.00,6813.00,6905.00,6955.00
Ethanol,4568.00,5021.00,5011.00,4648.00,5134.00,5200.00,5224.00,5432.00,5605.00,5378.00,4857.00,5033.00,5326.00,5176.00,5478.00,5436.00,5550.00,5600.00
Domestic Total,11079.00,11220.00,279.02,10379.00,11537.00,11883.00,11766.00,12356.00,12360.00,12223.00,12185.00,12068.00,12484.00,12045.00,12673.00,12267.00,13255.00,12905.00


... (10 outputs left out)


Exports


Marketing Year 2026/27,May-26,Jun-26,Jul-26,Aug-26,Sep-26
WASDE Item,,,,,
Exports,3150.00,3150.00,3200.00,3275.00,3275.00



Total Use & Ending Stocks


Marketing Year 2026/27,May-26,Jun-26,Jul-26,Aug-26,Sep-26
WASDE Item,,,,,
Total Use,16205.00,16205.00,16255.00,16330.00,16180.00
Ending Stocks,1957.00,1960.00,1790.00,41.98,39.79
Stocks-to-Use,0.12,0.12,0.11,0.10,0.10
Avg. Farm Price,4.40,4.40,4.40,4.50,4.80


In [ ]:

# ============================================================
# 10. RUN
# ============================================================

wasde = prepare_wasde_data(df)

corn_us = select_wasde_data(
    wasde,
    commodity="corn",
    country="united states",
)


# Inspect available raw identifiers.
# print("Available raw WASDE items:")
# print(sorted(corn_us["snd_item"].dropna().unique()))

# # Verify the 32-item mapping against the selected dataset.
# raw_items = set(corn_us["snd_item"].dropna().unique())
# mapped_items = set(WASDE_ITEM_LABELS)
# section_items = {
#     item
#     for items in WASDE_SECTIONS.values()
#     for item in items
# }

# print("\nMapping checks")
# print("Raw items in dataset:", len(raw_items))
# print("Keys in display mapping:", len(mapped_items))
# print("Items in sections:", len(section_items))
# print("Missing display labels:", sorted(raw_items - mapped_items))
# print("Missing from sections:", sorted(raw_items - section_items))
# print("Section identifiers not in dataset:", sorted(section_items - raw_items))

# ------------------------------------------------------------
# YEARLY BALANCE SHEET
# ------------------------------------------------------------

yearly_table = build_yearly_table(corn_us)

print("\nYEARLY BALANCE SHEET")

for section_name, section_table in split_wasde_table(yearly_table).items():
    print(f"\n{section_name}")
    display(style_wasde_table(section_table))

# ------------------------------------------------------------
# MONTHLY BALANCE SHEET
# ------------------------------------------------------------

monthly_table = build_monthly_table(
    corn_us,
    crop_year=2026,
)

print("\nMONTHLY BALANCE SHEET — 2026 CROP YEAR")

for section_name, section_table in split_wasde_table(monthly_table).items():
    print(f"\n{section_name}")
    display(style_wasde_table(section_table))


In [ ]:
import streamlit as st

from src.processes.process_wasde import load_wasde
from src.filters.wasde import filter_wasde_data
from src.tables.wasde import (
    build_yearly_table,
    split_wasde_table,
    style_wasde_table,
)


def render_wasde_view():
    """Render the WASDE page."""

    st.title("WASDE Balance Sheet")

    # --------------------------------------------------
    # 1. LOAD DATA
    # --------------------------------------------------
    df = load_wasde()

    if df.empty:
        st.warning("No WASDE data available.")
        return

    # --------------------------------------------------
    # 2. FILTER OPTIONS
    # --------------------------------------------------
    st.subheader("Filters")

    col1, col2, col3 = st.columns(3)

    with col1:
        commodities = sorted(
            df["commodity"].dropna().unique().tolist()
        )

        commodity = st.selectbox(
            "Commodity",
            commodities,
            key="wasde_commodity",
        )

    # Restrict country options to the selected commodity.
    commodity_df = df[
        df["commodity"].eq(commodity)
    ]

    with col2:
        countries = sorted(
            commodity_df["country"].dropna().unique().tolist()
        )

        country = st.selectbox(
            "Country",
            countries,
            key="wasde_country",
        )

    # Restrict table options to the selected commodity/country.
    country_df = commodity_df[
        commodity_df["country"].eq(country)
    ]

    with col3:
        if "table" in country_df.columns:
            tables = sorted(
                country_df["table"].dropna().unique().tolist()
            )
        else:
            tables = []

        table = st.selectbox(
            "WASDE Table",
            ["All"] + tables,
            key="wasde_table",
        )

    # --------------------------------------------------
    # 3. OPTIONAL FILTERS
    # --------------------------------------------------
    col4, col5, col6 = st.columns(3)

    with col4:
        wheat_class = None

        if "wheat_class" in country_df.columns:
            wheat_classes = sorted(
                country_df["wheat_class"]
                .dropna()
                .unique()
                .tolist()
            )

            if wheat_classes:
                selected_class = st.selectbox(
                    "Wheat Class",
                    ["All"] + wheat_classes,
                    key="wasde_wheat_class",
                )

                if selected_class != "All":
                    wheat_class = selected_class

    with col5:
        crop_status = None

        if "crop" in country_df.columns:
            crop_statuses = sorted(
                country_df["crop"].dropna().unique().tolist()
            )

            if crop_statuses:
                selected_crop = st.selectbox(
                    "Crop",
                    ["All"] + crop_statuses,
                    key="wasde_crop",
                )

                if selected_crop != "All":
                    crop_status = selected_crop

    # --------------------------------------------------
    # 4. APPLY FILTERS
    # --------------------------------------------------
    filtered = filter_wasde_data(
        df,
        commodity=commodity,
        country=country,
        table=None if table == "All" else table,
        wheat_class=wheat_class,
        crop_status=crop_status,
    )

    if filtered.empty:
        st.warning("No data matches the selected filters.")
        return

    # --------------------------------------------------
    # 5. DISPLAY DATA SUMMARY
    # --------------------------------------------------
    st.divider()

    latest_report = filtered["report_date"].max()

    metric1, metric2, metric3 = st.columns(3)

    metric1.metric("Commodity", commodity.title())
    metric2.metric("Country", country.title())

    metric3.metric(
        "Latest Report",
        latest_report.strftime("%d %b %Y")
        if not __import__("pandas").isna(latest_report)
        else "N/A",
    )

    # --------------------------------------------------
    # 6. BUILD YEARLY TABLE
    # --------------------------------------------------
    st.subheader("Yearly Balance Sheet")

    yearly_table = build_yearly_table(filtered)

    if yearly_table.empty:
        st.info("No yearly balance-sheet data available.")
        return

    # Split into WASDE sections.
    sections = split_wasde_table(yearly_table)

    # Display each section.
    if isinstance(sections, dict):
        for section_name, section_table in sections.items():
            if section_table is not None and not section_table.empty:
                st.markdown(f"#### {section_name.title()}")
                st.dataframe(
                    style_wasde_table(section_table),
                    use_container_width=True,
                )
    else:
        # Fallback if split_wasde_table returns one DataFrame.
        st.dataframe(
            style_wasde_table(sections),
            use_container_width=True,
        )


ModuleNotFoundError: No module named 'src'

In [8]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Data directory exists:", DATA_DIR.exists())
print("WASDE file exists:", (DATA_DIR / "wasde_processed_all_historicals.csv").exists())

Project root: /Users/jason/Desktop/Intern/agri_dashboard
Data directory exists: True
WASDE file exists: True


In [7]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/jason/Desktop/Intern/agri_dashboard")
DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Data directory exists:", DATA_DIR.exists())
print("WASDE file exists:", (DATA_DIR / "wasde_processed_all_historicals.csv").exists())
print("Files:", list(DATA_DIR.glob("*")) if DATA_DIR.exists() else "Directory not found")

Data directory exists: True
WASDE file exists: True
Files: [PosixPath('/Users/jason/Desktop/Intern/agri_dashboard/data/processed/wasde_processed_all_historicals.csv'), PosixPath('/Users/jason/Desktop/Intern/agri_dashboard/data/processed/cftc_data.txt')]


In [ ]:
import importlib
import src.processes.process_wasde as loaders

importlib.reload(loaders)

df = loaders.load_wasde()

print(df.shape)
display(df.head())

2026-10-14 11:22:56.248 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-14 11:22:56.250 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


(202960, 15)


,report_date,day,month,year,value,commodity,country,snd_item,wheat_class,crop,table,crop_year,month_offset,year_off,month_num
0,2024-01-12,12,1,2024,24.0,wheat,united states,beginning stocks,durum,oc,bc,2022,2_01,2,1
1,2024-01-12,12,1,2024,88.0,wheat,united states,domestic use,durum,oc,bc,2022,2_01,2,1
2,2024-01-12,12,1,2024,28.0,wheat,united states,ending stocks,durum,oc,bc,2022,2_01,2,1
3,2024-01-12,12,1,2024,23.0,wheat,united states,exports,durum,oc,bc,2022,2_01,2,1
4,2024-01-12,12,1,2024,0.0,wheat,united states,feed residual,durum,oc,bc,2022,2_01,2,1


In [56]:
years= [2025,2024,2023]
months = ["0_09","0_08","0_07"]
df_test=df[(df["commodity"]== "barley")&(df["table"]== "us")].copy()
df_test[(df_test["crop_year"].isin(years))&
        (df_test["month_offset"].isin(months))]



,report_date,day,month,year,value,commodity,country,snd_item,wheat_class,crop,table,crop_year,month_offset,year_off,month_num
172356,2025-07-11,11,7,2025,2.400000,barley,united states,planted area,<NA>,nc,us,2025,0_07,0,7
172358,2025-07-11,11,7,2025,1.900000,barley,united states,harvested area,<NA>,nc,us,2025,0_07,0,7
172360,2025-07-11,11,7,2025,77.100000,barley,united states,yield,<NA>,nc,us,2025,0_07,0,7
172362,2025-07-11,11,7,2025,70.000000,barley,united states,beginning stocks,<NA>,nc,us,2025,0_07,0,7
172364,2025-07-11,11,7,2025,148.000000,barley,united states,production,<NA>,nc,us,2025,0_07,0,7
172366,2025-07-11,11,7,2025,9.000000,barley,united states,imports,<NA>,nc,us,2025,0_07,0,7
172368,2025-07-11,11,7,2025,226.000000,barley,united states,supply total,<NA>,nc,us,2025,0_07,0,7
172370,2025-07-11,11,7,2025,30.000000,barley,united states,feed residual,<NA>,nc,us,2025,0_07,0,7
172372,2025-07-11,11,7,2025,115.000000,barley,united states,food seed industrial,<NA>,nc,us,2025,0_07,0,7
172374,2025-07-11,11,7,2025,145.000000,barley,united states,total domestic,<NA>,nc,us,2025,0_07,0,7


In [54]:
df_barley=df[(df["commodity"]== "barley")]
check = df_barley[(df_barley["crop_year"].isin(years))]
check["month_offset"].unique()

<ArrowStringArray>
['1_10', '0_10', '1_11', '0_11', '1_12', '0_12', '2_01', '1_01', '2_02',
 '1_02', '2_03', '1_03', '2_04', '1_04', '1_05', '0_05', '1_06', '0_06',
 '1_07', '0_07', '1_08', '0_08', '1_09', '0_09']
Length: 24, dtype: str